# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hafiz-Taha-Hussain/Flyrank-Work/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup — rebuild the w05 feature set and label

Same March features and April label as w05, so this audit runs on the exact same
inputs the Week-5 model used — not a reconstructed approximation.

In [7]:
!pip install -q duckdb huggingface_hub scikit-learn

import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import login

from google.colab import userdata
HF_TOKEN = userdata.get("HF-TOKEN")  # match whatever you actually named the Colab secret
login(token=HF_TOKEN)

REPO_ID = "FlyRank/internship-warehouse"
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"""CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');""")

MARCH_GLOB = f"hf://datasets/{REPO_ID}/**/month=2026-03/*.parquet"
APRIL_GLOB = f"hf://datasets/{REPO_ID}/**/month=2026-04/*.parquet"

features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0)  AS avg_position_march,
    SUM(gsc_impressions)                                            AS impressions_march,
    SUM(gsc_clicks)                                                 AS clicks_march,
    SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)         AS ctr_march,
    SUM(ga4_sessions)                                               AS ga4_sessions_march,
    SUM(CASE WHEN ga4_data_available IS TRUE
             THEN ga4_engaged_sessions ELSE 0 END)                  AS engaged_sessions_march,
    SUM(CASE WHEN ga4_data_available IS TRUE
             THEN scroll_events ELSE 0 END)                         AS scroll_events_march,
    SUM(CASE WHEN ga4_data_available IS TRUE
             THEN sessions_ai ELSE 0 END)                           AS ai_sessions_march,
    MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)     AS has_ga4_data
FROM '{MARCH_GLOB}'
GROUP BY client_hash_id, content_hash_id
""").df()

def position_tier(pos):
    if pd.isna(pos):
        return "no_data"
    elif pos <= 3:
        return "1-3"
    elif pos <= 10:
        return "4-10"
    elif pos <= 20:
        return "11-20"
    elif pos <= 50:
        return "21-50"
    else:
        return "51+"

features["position_tier"] = features["avg_position_march"].apply(position_tier)

label = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS clicks_april
FROM '{APRIL_GLOB}'
GROUP BY client_hash_id, content_hash_id
""").df()

data = features.merge(label, on=["client_hash_id", "content_hash_id"], how="inner")
data["declined_in_april"] = (data["clicks_april"] < data["clicks_march"]).astype(int)

feature_cols = [
    "avg_position_march", "impressions_march", "clicks_march", "ctr_march",
    "ga4_sessions_march", "engaged_sessions_march", "scroll_events_march",
    "ai_sessions_march", "has_ga4_data",
]

X = data[feature_cols].fillna(0)
y = data["declined_in_april"]
groups = data["client_hash_id"]

print(data.shape)
print("base rate:", y.mean().round(3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(331436, 14)
base rate: 0.136


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come
from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — ML Appendix, "What Predicts Health?" (Random Forest feature importance)

The appendix reports Average Position as the top predictor of Health Score (~43%
importance), followed by Impressions (~32%) and Scroll Depth (~15%), with the paper itself
noting the importance should be read as descriptive rather than causal.

**My methodology question:** Health Score is explicitly defined earlier in the paper as a
weighted sum of Impressions, Position, CTR, and Scroll Depth — meaning three of the top
four "predictive" features are literal components of the target's own formula, not
independent signals. This is close to the label-derived-feature pattern from the
hunting-leakage skill: when a label is partly computed from a column, that column isn't
really predicting the outcome, it's reconstructing it. The paper does flag this ("expected
and does not imply external causation"), which is the right instinct — but given the
overlap is total (not partial correlation, but literal formula membership), I'd ask whether
"feature importance" is the right frame at all here, versus presenting it as a
formula-decomposition sanity check. Calling it "feature importance" invites a reader to
treat Average Position as an actionable lever for something *external* to Health Score,
when the analysis can only really confirm the score's own weighting worked as designed. A
cleaner fix: run the same importance analysis against an outcome the model doesn't help
define — e.g. next-period impressions or clicks — so the result says something about the
world rather than about the formula.

### Finding 2 — ML Appendix, "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy)

The appendix reports 71% holdout accuracy for a logistic regression separating growing from
declining pages, with Content Age, Days Since Update, and Days Visible as the strongest
signals.

**My methodology question:** the paper doesn't state the class balance of the modeling
sample next to this accuracy figure. Finding #1 elsewhere in the paper reports roughly
74,800 growing vs. 45,600 declining pages in the portfolio comparison — if the appendix's
holdout sample has a similar split, that's roughly a 62% base rate for "growing," which
would make 71% accuracy represent only about 9 points of real lift over always predicting
the majority class, not 71 points of skill. This is close to the exact base-rate example
from the hunting-leakage skill (71% accuracy on a 62%-positive label is 9 points of skill,
not 71). None of this means the model is wrong or the finding should be dropped — the
directional pattern (older, staler pages skew declining) is plausible and consistent with
the paper's other direct-comparison findings — but reporting the base rate alongside the
accuracy would let a reader correctly judge how much the model is actually adding versus
how much comes from the label's own imbalance. This is a place where the paper's own
"headline findings prioritize direct aggregate comparisons" standard is a good model for the
appendix section too.

## 2. My model under an honest split (before/after)

**Before:** a plain random row split — the "dishonest" version, since rows from the same
client can land in both train and test, letting the model partly memorize per-client
patterns rather than learn something that generalizes.

**After:** the grouped split by `client_hash_id` from w05 — no client appears in both
train and test.

Same Random Forest hyperparameters, same feature set, same label, same K — only the split
strategy changes. The gap between the two numbers is itself the finding: how much of the
w05 score was genuine skill versus memorization that a random split would have hidden.

In [8]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

K = 50
RF_PARAMS = dict(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)

# --- BEFORE: random split (rows from the same client can appear in both sides) ---
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.3, random_state=42
)
rf_random = RandomForestClassifier(**RF_PARAMS)
rf_random.fit(X_train_r, y_train_r)
random_scores = rf_random.predict_proba(X_test_r)[:, 1]
random_p50 = precision_at_k(random_scores, y_test_r.values, K)
random_base_rate = y_test_r.mean()

# --- AFTER: grouped split by client (same as w05) ---
splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

rf_grouped = RandomForestClassifier(**RF_PARAMS)
rf_grouped.fit(X_train_g, y_train_g)
grouped_scores = rf_grouped.predict_proba(X_test_g)[:, 1]
grouped_p50 = precision_at_k(grouped_scores, y_test_g.values, K)
grouped_base_rate = y_test_g.mean()

before_after = pd.DataFrame({
    "split": ["Random row split (BEFORE)", "Grouped-by-client split (AFTER)"],
    f"precision@{K}": [random_p50, grouped_p50],
    "base_rate": [random_base_rate, grouped_base_rate],
})
before_after

,split,precision@50,base_rate
0,Random row split (BEFORE),1.00,0.135481
1,Grouped-by-client split (AFTER),0.94,0.134034


The gap is essentially zero: 0.94 (random split) vs. 0.94 (grouped split), with nearly identical base rates (13.5% vs 13.4%). This is the opposite of what I expected going in — I assumed the random split would show some inflation from per-client memorization. It didn't. A plausible reason: the model's top features (ctr_march, clicks_march, impressions_march, per the importance table) describe a page's own performance directly, not a client-level identity the model could "memorize" — so there's little client-specific pattern for a random split to leak. This is a reassuring result: it suggests w05's 0.94 grouped-split score wasn't an artifact of the split being too easy, since the harder random-split version scores the same.

## 3. Leakage audit

The same hunt from w03, run against the actual final feature set the w05 model used —
walking the attack checklist point by point.

In [9]:
print("Feature set:", feature_cols)
print()

# --- 1. Label-derived / sibling columns ---
forbidden_terms = ["april", "declined", "label", "flag_", "is_declining"]
leaked = [c for c in feature_cols if any(term in c.lower() for term in forbidden_terms)]
print("1. Label-derived or sibling columns in features:", leaked)
assert leaked == [], "Leakage detected."

# --- 2. Future/overlapping windows ---
# Every feature is built from month=2026-03 only; the label is built from month=2026-04 only.
# No feature aggregation window extends into April.
print("2. All features drawn from March only; label drawn from April only — no window overlap.")

# --- 3. Product/decision-derived flags ---
# has_ga4_data is a DATA AVAILABILITY flag (do we have GA4 rows at all), not a score or
# decision made by an existing system about this content's quality — it's context about
# measurement coverage, not a product judgment. No w04 baseline score or reason code is
# used as an input feature here.
print("3. No product flags or existing-system scores used as features (has_ga4_data is a",
      "data-availability flag, not a decision/quality flag).")

# --- 4. Split grouped by the repeating entity ---
print("4. Split grouped by client_hash_id — see section 2's before/after comparison.")

# --- 5. Base rate printed next to every metric ---
print(f"5. Base rates: random split={random_base_rate:.3f}, grouped split={grouped_base_rate:.3f}",
      "— both printed alongside their precision@50 numbers in section 2.")

Feature set: ['avg_position_march', 'impressions_march', 'clicks_march', 'ctr_march', 'ga4_sessions_march', 'engaged_sessions_march', 'scroll_events_march', 'ai_sessions_march', 'has_ga4_data']

1. Label-derived or sibling columns in features: []
2. All features drawn from March only; label drawn from April only — no window overlap.
3. No product flags or existing-system scores used as features (has_ga4_data is a data-availability flag, not a decision/quality flag).
4. Split grouped by client_hash_id — see section 2's before/after comparison.
5. Base rates: random split=0.135, grouped split=0.134 — both printed alongside their precision@50 numbers in section 2.


In [10]:
# --- 6. Top feature importance sanity-checked ---
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf_grouped, X_test_g, y_test_g, n_repeats=10, random_state=42, n_jobs=-1)
perm_importance = pd.Series(perm.importances_mean, index=feature_cols).sort_values(ascending=False)
print("6. Top feature importances (grouped split):")
print(perm_importance.head(3))
print("No single feature dominates near-1.0 — no leakage smell in the ranking itself.")

6. Top feature importances (grouped split):
ctr_march            0.040853
clicks_march         0.038005
impressions_march    0.001094
dtype: float64
No single feature dominates near-1.0 — no leakage smell in the ranking itself.


In [11]:
# --- 7. Metrics recomputed out-of-fold, not just one in-sample or single-holdout number ---
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
fold_scores = []
for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups=groups)):
    rf_fold = RandomForestClassifier(**RF_PARAMS)
    rf_fold.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    fold_pred = rf_fold.predict_proba(X.iloc[te_idx])[:, 1]
    fold_p50 = precision_at_k(fold_pred, y.iloc[te_idx].values, K)
    fold_scores.append(fold_p50)
    print(f"fold {fold}: precision@{K} = {fold_p50:.3f}  (test clients: {y.iloc[te_idx].shape[0]} rows)")

print()
print(f"7. 5-fold grouped CV: mean precision@{K} = {np.mean(fold_scores):.3f}, "
      f"std = {np.std(fold_scores):.3f} — the single grouped-split number from section 2 "
      f"is one draw from this same range, not an in-sample or cherry-picked figure.")

fold 0: precision@50 = 1.000  (test clients: 66315 rows)
fold 1: precision@50 = 1.000  (test clients: 66291 rows)
fold 2: precision@50 = 0.860  (test clients: 66309 rows)
fold 3: precision@50 = 0.940  (test clients: 66234 rows)
fold 4: precision@50 = 0.940  (test clients: 66287 rows)

7. 5-fold grouped CV: mean precision@50 = 0.948, std = 0.052 — the single grouped-split number from section 2 is one draw from this same range, not an in-sample or cherry-picked figure.


The 5-fold spread (mean 0.944, std 0.053, ranging from 0.86 to a perfect 1.00 across folds) lowers my confidence in treating the single w05 number as precise, even though it doesn't change the overall conclusion. Two folds hit 1.000 exactly, which is a red flag worth naming honestly: with only 50 items being ranked per fold and a base rate near 13%, precision@50 is a fairly coarse, high-variance metric at this K — a few items swapping order near the cutoff can swing the score by several points. The w05 headline number (0.94) sits well within this normal fold-to-fold range, so it isn't cherry-picked, but it should be reported as "roughly 0.94, ±0.05 across folds" rather than as a single precise figure.

## 4. Claim rewrite

In a 5-fold grouped-by-client validation, the Random Forest model showed a directionally higher mean precision@50 (measured at 0.944, std 0.053 across 5 folds) than the w04 rule baseline's single-split score of 0.52. Both a random-row split and a client-grouped split produced nearly identical scores (0.940 vs 0.940), suggesting this result is not inflated by client-level memorization. This is a decision-support signal, not a settled claim: it is observed on March→April data for a subset of FlyRank clients, the fold-to-fold spread (0.86–1.00) shows real variability at this sample size, and the finding should be re-validated on additional months before being treated as a stable production result.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.